[Reference](https://medium.com/@piyushagni5/minimalist-ai-workflows-with-lcel-langchain-expression-language-e9b7a3b8d15d)

In [1]:
# # Install all required packages
!pip install -qU \
    langchain==0.3.26 \
    langchain-google-genai==2.1.5 \
    langchain-huggingface==0.3.0 \
    langchain-core==0.3.66 \
    python-dotenv==1.1.1  \
    faiss-cpu==1.11.0 \
    sentence-transformers==4.1.0 \
    langchain-community==0.3.26 \
    torch==2.4.1 \
    torchvision==0.19.1

from dotenv import load_dotenv

### crete a .env and put your model API key (eg, GOOGLE_API_KEY=xxxxxx)

load_dotenv()

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Give me a short report about {topic}"
)

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
output_parser = StrOutputParser()

In [3]:
from langchain.chains import LLMChain

chain = LLMChain(
    prompt=prompt,
    llm=model,
    output_parser=output_parser
)

result = chain.run(topic="Computer Vision")
print(result)

In [4]:
lcel_chain = prompt | model | output_parser
result = lcel_chain.invoke({"topic": "Artificial Intelligence"})
print(result)

# How the Pipe Operator Works


In [5]:
class Wrapper:
    def __init__(self, value):
        self.value = value

    def __or__(self, func):
        return func(self.value)

def double(x):
    return x * 2

result = Wrapper(5) | double
print(result)  # Output: 10

10


In [6]:
w = Wrapper(5)
result = w | double

In [7]:
class Runnable:
    def __init__(self, func):
        self.func = func

    def __or__(self, other):
        def chained(x):
            return other(self.func(x))
        return Runnable(chained)

    def __call__(self, x):
        return self.func(x)

In [8]:
def add_five(x):
    return x + 5

def multiply_by_two(x):
    return x * 2

add_five = Runnable(add_five)
multiply_by_two = Runnable(multiply_by_two)

chain = add_five | multiply_by_two
print(chain(3))  # Output: 16

16


# Runnables in LangChain

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.prompts import ChatPromptTemplate

# Initialize embedding model
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create FAISS vector stores instead of DocArrayInMemorySearch
vecstore_a = FAISS.from_texts(
    ["James' birthday is the 7th December"],
    embedding=embedding
)
vecstore_b = FAISS.from_texts(
    ["James was born in 1994"],
    embedding=embedding
)

In [11]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# Create retrievers
retriever_a = vecstore_a.as_retriever()
retriever_b = vecstore_b.as_retriever()

# First approach - separate context fields
prompt = ChatPromptTemplate.from_template("""
Answer the question using context:

Context: {context}

Question: {question}

Answer:
""")

retrieval = RunnableParallel(
    {"context": retriever_a, "question": RunnablePassthrough()}
)

# Note: You'll need to define 'model' and 'output_parser' variables
chain = retrieval | prompt | model | output_parser
print(chain.invoke("when was James born?"))

In [12]:
print(chain.invoke("when year James was born?"))

In [13]:
# Second approach - both context fields
prompt = ChatPromptTemplate.from_template("""
Answer the question using context:

Context A: {context_a}

Context B: {context_b}

Question: {question}

Answer:
""")


retrieval = RunnableParallel({
    "context_a": retriever_a,
    "context_b": retriever_b,
    "question": RunnablePassthrough()
})

# Note: You'll need to define 'model' and 'output_parser' variables
chain = retrieval | prompt | model | output_parser
print(chain.invoke("when was James born?"))

# RunnableLambda for Custom Logic

In [14]:
from langchain_core.runnables import RunnableLambda

def add_five(x):
    return x + 5

def multiply_by_two(x):
    return x * 2

# wrap the functions with RunnableLambda
add_five = RunnableLambda(add_five)
multiply_by_two = RunnableLambda(multiply_by_two)

In [15]:
chain = add_five | multiply_by_two
chain.invoke(3) # output: 16

prompt_str = "Tell me an short fact about {topic}"
prompt = ChatPromptTemplate.from_template(prompt_str)

chain = prompt | model | output_parser
chain.invoke({"topic": "Artificial Intelligence"})

In [16]:
# Example 1: Format facts as bullet points
def format_as_bullet(text):
    return f"• {text.strip()}"

format_bullet = RunnableLambda(format_as_bullet)
chain = prompt | model | output_parser | format_bullet

chain.invoke({"topic": "Artificial Intelligence"})